# Algerian Lamiaceae Leaves — Dataset Preparation (v3, revision)
### Three-Source Integration: PlantNet-300K · MedLeaves · Medicinal Leaf Dataset

| Step | Task |
|------|------|
| 1 | Configure paths and revision options |
| 2 | Define target Algerian Lamiaceae species |
| 3 | Shared matching helpers |
| 4 | Source A — PlantNet-300K (Kaggle, JSON-guided) |
| 5 | Source B — MedLeaves (Kaggle, folder matching) |
| 6 | Source C — Medicinal Leaf Dataset (local) |
| 7 | Merge: identification level, exact + **near-duplicate** removal, **neutral pooled labels**, splits |
| 8 | Copy to unified output, metadata, audit sample |
| 9–13 | Statistics, charts, Table 1 |
| 14 / 14b | Disk check; leaf-organ quality audit (re-uses the v2 decisions) |
| 15–18 | Augmentation, undersampling, final statistics |

**Output:** `Algerian_Lamiaceae_v3/{train,val,test}/<class>/` + `metadata.csv` + `class_composition.csv` + duplicate logs

### What changed compared with v2 (reviewer remaining issues)

| Issue | Change |
|---|---|
| 6 — near-duplicate leakage | After SHA-256, every image is perceptual-hashed (pHash + dHash). Near-duplicate groups (resized, cropped, recompressed copies) keep **one** image **before** splitting, so no near-duplicate can sit in two splits. Groups whose copies carry different labels are removed. Log: `near_duplicates_removed.csv`. |
| 2 — pooled labels | Every image now records its **original identification** (`original_name`) and whether it matches the class species exactly (`match_level`). Classes that contain other species are renamed `<Genus>_pooled` (e.g. the old *Lamium flexuosum* class held 11 *Lamium* species; the old *Lavandula stoechas* class held 8 other *Lavandula* taxa; *Plectranthus* included *P. barbatus*). Composition table: `class_composition.csv`. Unidentified *Lavandula spp.* images are dropped, because they could belong to either *Lavandula* class. |
| 5 — sample audit | The 5% audit sample re-uses the 1,201 decisions already made on v2 (matched by source image), so only the remaining images must be reviewed — the whole sample can now be completed. |
| Safety | The old Step 1 deleted the whole output folder, including `results/`. Now only `train/ val/ test/` and generated backups are reset; v2 is never modified. |

**After running this notebook, every downstream notebook (01–16) must be re-run with `DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"`** — class names and image counts change (update `EXPECTED_SPLIT_COUNTS` in Notebook 14 from the Step 18 table).

## Step 1 — Configure Paths & Install Dependencies

In [ ]:

PLANTNET_ROOT = "/home/ali/Desktop/dataset/plantnet_300K"

MEDLEAVES_ROOT = "/home/ali/Desktop/dataset/archive(6)/MedLeaves/MedLeaves"

MEDLEAF_ROOT = "/home/ali/Desktop/dataset/Medicinal Leaf dataset"

# v3 = rebuilt dataset (near-duplicate screening + neutral pooled labels).
# v2 is NOT touched: it stays as the record of the submitted results and is
# used only to re-use the manual audit decisions already made (Step 14b).
OUTPUT_ROOT   = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
PREVIOUS_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v2"

# ---- Revision options (reviewer remaining issues 2 and 6) ----
NEAR_DUP_SCREENING    = True   # perceptual-hash screening after SHA-256 (issue 6)
PHASH_T, DHASH_T      = 8, 12  # 64-bit hashes: near-duplicate if pHash <= 8 AND dHash <= 12
NEUTRAL_POOLED_LABELS = True   # classes containing other species -> "<Genus>_pooled" (issue 2)
EXCLUDE_UNIDENTIFIED  = True   # drop "Genus spp." images when that genus has >1 class (cannot be assigned)

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pillow", "tqdm", "pyyaml", "imagehash", "--quiet"])

import os, re, json, shutil, csv, random
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

print("Path check:")
for name, p in [("PlantNet-300K",  PLANTNET_ROOT),
                ("MedLeaves",      MEDLEAVES_ROOT),
                ("MedLeaf local",  MEDLEAF_ROOT),
                ("Previous (v2)",  PREVIOUS_ROOT)]:
    status = "FOUND" if os.path.exists(p) else "NOT FOUND -- check path"
    print(f"  [{status}]  {name}: {p}")

# Reset ONLY what this notebook generates. The previous version deleted the
# whole OUTPUT_ROOT, which also wiped results/ (classifier, CNN and docking
# outputs written there by the other notebooks). results/ is never touched now.
GENERATED = ["train", "val", "test", "train_lamium_backup", "_removed_flagged_images"]
os.makedirs(OUTPUT_ROOT, exist_ok=True)
for sub in GENERATED:
    p = os.path.join(OUTPUT_ROOT, sub)
    if os.path.isdir(p):
        print(f"[RESET] removing previous {p}")
        shutil.rmtree(p)
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_ROOT, split), exist_ok=True)
print(f"\nOutput: {OUTPUT_ROOT}  (train/val/test clean; results/ and caches kept)")


In [ ]:
import os, json
import pandas as pd

all_species_records = []

plantnet_json = os.path.join(PLANTNET_ROOT, "plantnet300K_species_id_2_name.json")
with open(plantnet_json, "r", encoding="utf-8") as f:
    id2name = json.load(f)

print(f"PlantNet-300K: {len(id2name)} total species in taxonomy file")

plantnet_img_root = os.path.join(PLANTNET_ROOT, "images")
for species_id, species_name in id2name.items():
    n_imgs = 0
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(plantnet_img_root, split, species_id)
        if os.path.isdir(split_dir):
            n_imgs += len([f for f in os.listdir(split_dir)
                           if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    all_species_records.append({
        "source": "PlantNet-300K",
        "species_id": species_id,
        "class_name": species_name,
        "n_images": n_imgs,
    })

medleaves_classes = {}
for split in ["train", "val", "test"]:
    split_dir = os.path.join(MEDLEAVES_ROOT, split)
    if not os.path.isdir(split_dir):
        continue
    for class_folder in os.listdir(split_dir):
        class_path = os.path.join(split_dir, class_folder)
        if not os.path.isdir(class_path):
            continue
        n_imgs = len([f for f in os.listdir(class_path)
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        medleaves_classes[class_folder] = medleaves_classes.get(class_folder, 0) + n_imgs

for class_name, n_imgs in medleaves_classes.items():
    all_species_records.append({
        "source": "MedLeaves",
        "species_id": None,
        "class_name": class_name,
        "n_images": n_imgs,
    })

print(f"MedLeaves: {len(medleaves_classes)} total class folders")

medleaf_classes = {}
for class_folder in os.listdir(MEDLEAF_ROOT):
    class_path = os.path.join(MEDLEAF_ROOT, class_folder)
    if not os.path.isdir(class_path):
        continue
    n_imgs = len([f for f in os.listdir(class_path)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    medleaf_classes[class_folder] = n_imgs

for class_name, n_imgs in medleaf_classes.items():
    all_species_records.append({
        "source": "Medicinal Leaf Dataset",
        "species_id": None,
        "class_name": class_name,
        "n_images": n_imgs,
    })

print(f"Medicinal Leaf Dataset: {len(medleaf_classes)} total class folders")

df_all_species = pd.DataFrame(all_species_records)
df_all_species = df_all_species.sort_values(["source", "class_name"]).reset_index(drop=True)

print(f"\nTOTAL raw class/species entries across all sources: {len(df_all_species)}")
print(df_all_species.groupby("source")["n_images"].agg(["count", "sum"]))

out_csv = os.path.join(OUTPUT_ROOT, "all_species_inventory_RAW.csv")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
df_all_species.to_csv(out_csv, index=False)
print(f"\nFull raw inventory saved -> {out_csv}")

pd.set_option("display.max_rows", None)
print("\n=== FULL SPECIES/CLASS INVENTORY ===")
print(df_all_species.to_string(index=False))

## Step 2 — Define Target Algerian Lamiaceae Species

In [ ]:

ALGERIAN_LAMIACEAE = {
    "Mentha_spicata":          {"genus":"Mentha",      "species":"Mentha spicata L.",
                                "arabic":"النعناع",     "fidelity":"high",
                                "region":"All Algeria — #2 cited in Adrar (39 citations)",
                                "uses":"Digestive, cold, antispasmodic, IBS"},
    "Mentha_pulegium":         {"genus":"Mentha",      "species":"Mentha pulegium L.",
                                "arabic":"الفليو",      "fidelity":"high",
                                "region":"North Algeria, Kabylia",
                                "uses":"RTD: chest/lung, UTI, infertility, insomnia"},
    "Mentha_rotundifolia":     {"genus":"Mentha",      "species":"Mentha rotundifolia (L.) Huds.",
                                "arabic":"تيمرصاد",     "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"IBS, gastrointestinal"},
    "Mentha_arvensis":         {"genus":"Mentha",      "species":"Mentha arvensis L.",
                                "arabic":"النعناع البري","fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Cardiovascular, infertility, chest, IBS"},
    "Mentha_aquatica":         {"genus":"Mentha",      "species":"Mentha aquatica L.",
                                "arabic":"حبق الماء",   "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Anxiety, nervous system"},
    "Origanum_vulgare":        {"genus":"Origanum",    "species":"Origanum vulgare L.",
                                "arabic":"الزعتر البري","fidelity":"high",
                                "region":"Adrar #1 cited (51 citations); Sahara",
                                "uses":"Respiratory, antibacterial, antifungal"},
    "Origanum_majorana":       {"genus":"Origanum",    "species":"Origanum majorana L.",
                                "arabic":"المردقوش",    "fidelity":"high (FL 50% SD)",
                                "region":"North Algeria",
                                "uses":"Skin: limb swelling; anti-inflammatory"},
    "Thymus_vulgaris":         {"genus":"Thymus",      "species":"Thymus vulgaris L.",
                                "arabic":"الزعتر",      "fidelity":"FL 100% skin diseases",
                                "region":"All Algeria",
                                "uses":"Skin, IBS, UTI, fever, cancer"},
    "Thymus_capitatus":        {"genus":"Thymus",      "species":"Thymus capitatus (L.) Hoffmanns. & Link.",
                                "arabic":"الزعتر المقرنس","fidelity":"high",
                                "region":"Algerian high plains: Souk Ahras, Guelma",
                                "uses":"Antioxidant, antibacterial — Tamma et al. 2024"},
    "Marrubium_vulgare":       {"genus":"Marrubium",   "species":"Marrubium vulgare L.",
                                "arabic":"المريوت",     "fidelity":"FL 100% (cancer, RTD, GISD, SRP)",
                                "region":"All Algeria — highest FL across multiple categories",
                                "uses":"Cancer, respiratory, gastrointestinal, infertility"},
    "Rosmarinus_officinalis":  {"genus":"Rosmarinus",  "species":"Rosmarinus officinalis L.",
                                "arabic":"إكليل الجبل", "fidelity":"medium",
                                "region":"North Algeria, Kabylia",
                                "uses":"Cholesterol, IBS, jaundice, antioxidant"},
    "Salvia_officinalis":      {"genus":"Salvia",      "species":"Salvia officinalis L.",
                                "arabic":"المريمية",    "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Cholesterol, cardiovascular"},
    "Lavandula_stoechas":      {"genus":"Lavandula",   "species":"Lavandula stoechas L.",
                                "arabic":"الحلحال",     "fidelity":"medium",
                                "region":"All Algeria",
                                "uses":"Diabetes, cholesterol, kidney, UTI, blood purification"},
    "Lavandula_angustifolia":  {"genus":"Lavandula",   "species":"Lavandula angustifolia Mill.",
                                "arabic":"الخزامى",     "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Hemorrhoids"},
    "Ocimum_pooled":        {"genus":"Ocimum",      "species":"Ocimum basilicum L.",
                                "arabic":"الريحان",     "fidelity":"high",
                                "region":"All Algeria",
                                "uses":"Infertility, IBS, UTI, fever, dizziness, hypertension"},
    "Melissa_officinalis":     {"genus":"Melissa",     "species":"Melissa officinalis L.",
                                "arabic":"مليسا",       "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Insomnia, nervous system"},
    "Ajuga_iva":               {"genus":"Ajuga",       "species":"Ajuga iva (L.) Schreb.",
                                "arabic":"الشندقورة",   "fidelity":"medium",
                                "region":"Adrar #3 cited; all Algeria",
                                "uses":"Cholesterol, cardiovascular"},
    "Clinopodium_nepeta":      {"genus":"Clinopodium", "species":"Clinopodium nepeta (L.) Kuntze",
                                "arabic":"النابطة",     "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"IBS, diabetes, cholesterol, kidney, UTI"},
    "Teucrium_polium":         {"genus":"Teucrium",    "species":"Teucrium polium L.",
                                "arabic":"خياطة الجراح","fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Gastric ulcers"},
    "Teucrium_atratum":        {"genus":"Teucrium",    "species":"Teucrium atratum Pomel",
                                "arabic":"تيوكريوم",    "fidelity":"high (endemic)",
                                "region":"NE Algeria endemic — Sadou et al. 2025",
                                "uses":"Antioxidant, antibacterial"},
    "Teucrium_spinosum":       {"genus":"Teucrium",    "species":"Teucrium spinosum L.",
                                "arabic":"الجعدة",      "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Blood purification, ulcers, diabetes"},
    "Saccocalyx_satureioides": {"genus":"Saccocalyx",  "species":"Saccocalyx satureioides Coss. & Durieu",
                                "arabic":"البليزير",    "fidelity":"FL 100% ESD (endemic)",
                                "region":"High plains Algeria",
                                "uses":"Diabetes, stomach"},
    "Stachys_marrubiifolia":   {"genus":"Stachys",     "species":"Stachys marrubiifolia Viv.",
                                "arabic":"ستاكيس",      "fidelity":"high (endemic)",
                                "region":"Algeria endemic",
                                "uses":"Anti-inflammatory — Benbouazza et al. 2025"},
    "Lamium_flexuosum":        {"genus":"Lamium",      "species":"Lamium flexuosum Ten.",
                                "arabic":"لاميوم",      "fidelity":"high (endemic)",
                                "region":"Algeria endemic",
                                "uses":"Anti-inflammatory — Benbouazza et al. 2025"},
    "Vitex_agnus_castus":      {"genus":"Vitex",       "species":"Vitex agnus-castus L.",
                                "arabic":"كف مريم",     "fidelity":"medium",
                                "region":"North Algeria",
                                "uses":"Breast/uterine tumours, sciatica"},
    "Plectranthus_amboinicus": {"genus":"Plectranthus","species":"Plectranthus amboinicus (Lour.) Spreng.",
                                "arabic":"النعناع الهندي","fidelity":"medium",
                                "region":"Algeria (folk)",
                                "uses":"Respiratory, antibacterial, cough"},
    "Leucas_aspera":          {"genus":"Leucas",       "species":"Leucas aspera (Willd.) Link",
                                "arabic":"الضمران",     "fidelity":"medium",
                                "region":"North Algeria — folk use",
                                "uses":"Antimicrobial, anti-inflammatory, fever"},
}

LAMIACEAE_GENERA = {v["genus"] for v in ALGERIAN_LAMIACEAE.values()}

print(f"Target species : {len(ALGERIAN_LAMIACEAE)}")
print(f"Genera covered : {sorted(LAMIACEAE_GENERA)}")


## Step 3 — Shared Matching Helpers

In [ ]:

def match_to_label(name_str):
    """
    Given a scientific name or folder name, return (canonical_label, species_name)
    if it belongs to Algerian Lamiaceae, else (None, None).

    Handles:
      - Exact genus+epithet match  e.g. 'Mentha spicata'
      - Folder names with _/-      e.g. 'Mentha_spicata', 'mentha-spicata'
      - Genus-only match           e.g. 'Mentha' -> 'Mentha_spicata' (first match)
    """
    clean = name_str.replace("_", " ").replace("-", " ").strip()
    tokens = clean.split()
    if not tokens:
        return None, None

    genus = tokens[0].capitalize()
    if genus not in LAMIACEAE_GENERA:
        return None, None

    if len(tokens) >= 2:
        binomial = f"{tokens[0].capitalize()} {tokens[1].lower()}"
        for label, info in ALGERIAN_LAMIACEAE.items():
            sp_tokens = info["species"].split()
            sp_bin = f"{sp_tokens[0]} {sp_tokens[1].lower()}" if len(sp_tokens)>=2 else sp_tokens[0]
            if binomial.lower() == sp_bin.lower():
                return label, info["species"]

    for label, info in ALGERIAN_LAMIACEAE.items():
        if info["genus"] == genus:
            return label, info["species"]

    return f"{genus}_sp", f"{genus} sp."

print("Matching helpers defined.")


In [ ]:

from pathlib import Path

SEP = "=" * 65

print(SEP)
print("PLANTNET-300K — JSON species matched vs unmatched")
print(SEP)
pn_root = Path(PLANTNET_ROOT)
id2name_p = pn_root / "plantnet300K_species_id_2_name.json"
if not id2name_p.exists():
    id2name_p = pn_root.parent / "plantnet300K_species_id_2_name.json"
if id2name_p.exists():
    with open(id2name_p, encoding="utf-8") as f:
        id2name_all = json.load(f)
    matched_pn, unmatched_pn = [], []
    for sid, sname in id2name_all.items():
        lbl, _ = match_to_label(sname)
        if lbl:
            matched_pn.append((sid, sname, lbl))
        else:
            unmatched_pn.append(sname)
    print(f"Matched   : {len(matched_pn)}")
    for sid, name, lbl in sorted(matched_pn, key=lambda x: x[1]):
        print(f"  {sid:>8}  {name:<55} -> {lbl}")
    print(f"\nUnmatched Lamiaceae-like names (genus in our list but no match):")
    for name in sorted(unmatched_pn):
        first = name.split()[0] if name.split() else ""
        if first in LAMIACEAE_GENERA:
            print(f"  {name}")
else:
    print(f"[ERROR] JSON not found at {id2name_p}")

print()
print(SEP)
print("MEDLEAVES — all folder names")
print(SEP)
ml_root = Path(MEDLEAVES_ROOT)
if not ml_root.exists():
    print(f"NOT FOUND: {MEDLEAVES_ROOT}")
else:
    top = sorted([d for d in ml_root.iterdir() if d.is_dir()])
    top_names = {d.name.lower() for d in top}
    has_splits = bool(top_names & {"train","val","test"})
    print(f"Has train/val/test: {has_splits}")
    if has_splits:
        seen = set()
        for sd in top:
            if not sd.is_dir(): continue
            for d in sorted(sd.iterdir()):
                if d.is_dir() and d.name not in seen:
                    seen.add(d.name)
                    lbl, _ = match_to_label(d.name)
                    n = sum(1 for f in d.rglob("*")
                            if f.suffix.lower() in {".jpg",".jpeg",".png"})
                    status = f"-> {lbl}" if lbl else "NO MATCH"
                    print(f"  [{status:35}]  '{d.name}'  ({n} imgs)")
    else:
        for d in top:
            lbl, _ = match_to_label(d.name)
            n = sum(1 for f in d.rglob("*")
                    if f.suffix.lower() in {".jpg",".jpeg",".png"})
            status = f"-> {lbl}" if lbl else "NO MATCH"
            print(f"  [{status:35}]  '{d.name}'  ({n} imgs)")

print()
print(SEP)
print("MEDICINAL LEAF DATASET (local) — all folder names")
print(SEP)
lc_root = Path(MEDLEAF_ROOT)
if not lc_root.exists():
    print(f"NOT FOUND: {MEDLEAF_ROOT}")
else:
    for d in sorted(lc_root.iterdir()):
        if not d.is_dir(): continue
        lbl, _ = match_to_label(d.name)
        n = sum(1 for f in d.rglob("*")
                if f.suffix.lower() in {".jpg",".jpeg",".png"})
        status = f"-> {lbl}" if lbl else "NO MATCH"
        print(f"  [{status:35}]  '{d.name}'  ({n} imgs)")

print()
print("Paste ALL lines above into chat to get the updated matching dictionary.")


## Step 4 — Source A: PlantNet-300K (Kaggle)

In [ ]:

plantnet_records = []
plantnet_root    = Path(PLANTNET_ROOT)

def find_json(root, filename):
    """Search root/ and root/../ for a JSON file."""
    for candidate in [root / filename, root.parent / filename]:
        p = candidate.resolve()
        if p.exists():
            return p
    return None

id2name_path = find_json(plantnet_root, "plantnet300K_species_id_2_name.json")
meta_path    = find_json(plantnet_root, "plantnet_300K_metadata.json")

images_root = plantnet_root / "images"
if not images_root.exists():
    images_root = plantnet_root

if not plantnet_root.exists():
    print(f"[SKIP] PlantNet-300K root not found: {PLANTNET_ROOT}")
elif id2name_path is None:
    print(f"[ERROR] plantnet300K_species_id_2_name.json not found near {PLANTNET_ROOT}")
    print("       Files found in root:")
    for f in sorted(plantnet_root.iterdir()):
        print(f"  {f.name}")
else:
    print(f"JSON  : {id2name_path}")
    print(f"Images: {images_root}")

    with open(id2name_path, "r", encoding="utf-8") as f:
        id2name = json.load(f)
    print(f"PlantNet total species: {len(id2name)}")

    target_ids = {}
    for sid, sname in id2name.items():
        label, sp = match_to_label(sname)
        if label:
            target_ids[sid] = (label, sname)

    if not target_ids:
        print("[WARN] No Lamiaceae species matched in PlantNet JSON.")
        print("       First 20 entries — check naming convention:")
        for sid, name in list(id2name.items())[:20]:
            print(f"  {sid}: {name}")
    else:
        print(f"\nAlgerian Lamiaceae species found: {len(target_ids)}")
        for sid, (label, name) in sorted(target_ids.items(), key=lambda x: x[1][1]):
            print(f"  {sid:>8}  {name:<50} -> {label}")

    img_meta = {}
    if meta_path and meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if isinstance(raw, list):
            for e in raw:
                k = str(e.get("image_id") or e.get("id") or "")
                if k:
                    img_meta[k] = e
        elif isinstance(raw, dict):
            img_meta = {str(k): v for k, v in raw.items()}
        print(f"  Metadata: {len(img_meta):,} records (cc-by-sa authors in metadata.csv)")
    else:
        print("  [WARN] plantnet_300K_metadata.json not found — author info unavailable")

    found_splits = [s for s in ["train","val","test"] if (images_root/s).exists()]
    print(f"\nSplit folders found: {found_splits}")

    for split in found_splits:
        split_dir = images_root / split
        for sid, (label, sname) in target_ids.items():
            sp_dir = split_dir / str(sid)
            if not sp_dir.exists():
                continue
            imgs = [f for f in sp_dir.iterdir()
                    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
            for img in imgs:
                meta = img_meta.get(img.stem, {})
                plantnet_records.append({
                    "source":     "PlantNet-300K",
                    "label":      label,
                    "species":    sname,
                    "genus":      sname.split()[0],
                    "split":      split,
                    "src_path":   str(img),
                    "filename":   img.name,
                    "author":     meta.get("author", "unknown"),
                    "license":    meta.get("license", "cc-by-sa"),
                    "species_id": sid,
                })

cnt = Counter(r["label"] for r in plantnet_records)
print(f"\nPlantNet-300K  ->  {len(plantnet_records):,} Lamiaceae images collected")
for label, n in sorted(cnt.items()):
    print(f"  {label:<35} {n:>5}")


## Step 5 — Source B: MedLeaves (Kaggle)

In [ ]:

import yaml

medleaves_records = []
medleaves_root    = Path(MEDLEAVES_ROOT)

if not medleaves_root.exists():
    print(f"[SKIP] MedLeaves not found: {MEDLEAVES_ROOT}")
else:
    class_map = {}

    yaml_candidates = [
        medleaves_root / "data.yaml",
        medleaves_root / "dataset.yaml",
        medleaves_root.parent / "data.yaml",
    ]
    txt_candidates = [
        medleaves_root / "classes.txt",
        medleaves_root / "obj.names",
        medleaves_root / "names.txt",
    ]

    yaml_path = next((p for p in yaml_candidates if p.exists()), None)
    txt_path  = next((p for p in txt_candidates  if p.exists()), None)

    if yaml_path:
        print(f"Class file: {yaml_path}")
        try:
            with open(yaml_path, encoding="utf-8") as f:
                data_yaml = yaml.safe_load(f)
            names = data_yaml.get("names", [])
            if isinstance(names, list):
                class_map = {i: n for i, n in enumerate(names)}
            elif isinstance(names, dict):
                class_map = {int(k): v for k, v in names.items()}
        except Exception as e:
            print(f"  [WARN] Could not parse {yaml_path}: {e}")
    elif txt_path:
        print(f"Class file: {txt_path}")
        with open(txt_path, encoding="utf-8") as f:
            class_map = {i: ln.strip() for i, ln in enumerate(f) if ln.strip()}
    else:
        print("[WARN] No data.yaml / classes.txt found.")
        print("       Files in MedLeaves root:")
        for f in sorted(medleaves_root.iterdir()):
            print(f"  {f.name}")

    if class_map:
        print(f"\nClasses in dataset ({len(class_map)}):")
        for cid, name in sorted(class_map.items()):
            lbl, _ = match_to_label(name)
            status = f"-> {lbl}" if lbl else "not Lamiaceae"
            print(f"  {cid:3d}  {name:<45}  {status}")

        lamiaceae_ids = {}
        for cid, name in class_map.items():
            lbl, sp = match_to_label(name)
            if lbl:
                lamiaceae_ids[cid] = (lbl, sp)

        print(f"\nLamiaceae class IDs: {sorted(lamiaceae_ids.keys())}")

        if not lamiaceae_ids:
            print("[WARN] No Lamiaceae classes found in data.yaml.")
            print("       Paste the class list above into chat to fix the mapping.")
        else:
            split_map = {"train":"train", "valid":"val",
                         "val":"val", "test":"test"}

            for split_folder in sorted(medleaves_root.iterdir()):
                if not split_folder.is_dir(): continue
                split = split_map.get(split_folder.name.lower())
                if split is None: continue

                img_dir = split_folder / "images"
                lbl_dir = split_folder / "labels"

                if not img_dir.exists():
                    print(f"  [WARN] No images/ in {split_folder.name}/")
                    continue

                img_files = sorted([f for f in img_dir.iterdir()
                                    if f.suffix.lower() in
                                    {".jpg",".jpeg",".png"}])

                for img_path in img_files:
                    ann_path = lbl_dir / (img_path.stem + ".txt")
                    img_class_ids = set()

                    if ann_path.exists():
                        with open(ann_path) as af:
                            for line in af:
                                parts = line.strip().split()
                                if parts:
                                    try:
                                        img_class_ids.add(int(parts[0]))
                                    except ValueError:
                                        pass
                    else:
                        if len(lamiaceae_ids) == 1:
                            img_class_ids = set(lamiaceae_ids.keys())

                    matched_ids = img_class_ids & set(lamiaceae_ids.keys())
                    if not matched_ids:
                        continue

                    cid = sorted(matched_ids)[0]
                    label, sp_name = lamiaceae_ids[cid]

                    medleaves_records.append({
                        "source":     "MedLeaves",
                        "label":      label,
                        "species":    sp_name,
                        "genus":      sp_name.split()[0],
                        "split":      split,
                        "src_path":   str(img_path),
                        "filename":   img_path.name,
                        "author":     "MedLeaves dataset",
                        "license":    "see dataset license",
                        "species_id": str(cid),
                    })

    else:
        print("[INFO] No class map — collecting all images as unlabelled.")
        print("       Add a data.yaml with 'names:' list to enable filtering.")
        split_map = {"train":"train","valid":"val","val":"val","test":"test"}
        for split_folder in sorted(medleaves_root.iterdir()):
            if not split_folder.is_dir(): continue
            split = split_map.get(split_folder.name.lower())
            if split is None: continue
            img_dir = split_folder / "images"
            if not img_dir.exists(): continue
            for img_path in sorted(img_dir.iterdir()):
                if img_path.suffix.lower() not in {".jpg",".jpeg",".png"}: continue
                medleaves_records.append({
                    "source":     "MedLeaves",
                    "label":      "unknown_medleaves",
                    "species":    "unknown",
                    "genus":      "unknown",
                    "split":      split,
                    "src_path":   str(img_path),
                    "filename":   img_path.name,
                    "author":     "MedLeaves dataset",
                    "license":    "see dataset license",
                    "species_id": "",
                })

cnt = Counter(r["label"] for r in medleaves_records)
print(f"\nMedLeaves  ->  {len(medleaves_records):,} Lamiaceae images collected")
for label, n in sorted(cnt.items()):
    print(f"  {label:<35} {n:>5}")


## Step 6 — Source C: Medicinal Leaf Dataset (Local)

In [ ]:

COMMON_NAME_MAP = {
    "Mint":         ("Mentha_spicata",          "Mentha spicata L."),
    "Doddpathre":   ("Plectranthus_amboinicus",  "Plectranthus amboinicus (Lour.) Spreng."),
    "Thumbe":       ("Leucas_aspera",             "Leucas aspera (Willd.) Link"),
    "Tulasi":       ("Ocimum_pooled",          "Ocimum basilicum L."),
    "Basil":        ("Ocimum_pooled",          "Ocimum basilicum L."),
    "Rosemary":     ("Rosmarinus_officinalis",    "Rosmarinus officinalis L."),
    "Lavender":     ("Lavandula_stoechas",        "Lavandula stoechas L."),
    "Thyme":        ("Thymus_vulgaris",           "Thymus vulgaris L."),
    "Oregano":      ("Origanum_vulgare",          "Origanum vulgare L."),
    "Sage":         ("Salvia_officinalis",        "Salvia officinalis L."),
    "Marjoram":     ("Origanum_majorana",         "Origanum majorana L."),
    "Spearmint":    ("Mentha_spicata",            "Mentha spicata L."),
    "Peppermint":   ("Mentha_pulegium",           "Mentha pulegium L."),
}

medleaf_records = []
medleaf_root    = Path(MEDLEAF_ROOT)

if not medleaf_root.exists():
    print(f"[SKIP] Medicinal Leaf Dataset not found: {MEDLEAF_ROOT}")
else:
    all_class_dirs = sorted([d for d in medleaf_root.iterdir() if d.is_dir()])
    print(f"Medicinal Leaf Dataset : {medleaf_root}")
    print(f"Total class folders    : {len(all_class_dirs)}")

    matched_lc, skipped_lc = [], []

    for folder in all_class_dirs:
        fname = folder.name.strip()

        norm_map = {k.strip().lower(): v for k, v in COMMON_NAME_MAP.items()}
        entry = COMMON_NAME_MAP.get(fname) or norm_map.get(fname.strip().lower())

        if entry is None:
            lbl, sp = match_to_label(fname)
            if lbl:
                entry = (lbl, sp)

        if entry:
            matched_lc.append((folder, entry[0], entry[1]))
        else:
            skipped_lc.append(fname)

    if not matched_lc:
        print("\n[WARN] Still no Lamiaceae folders matched after common-name lookup.")
        print("       Folder names in dataset:")
        for name in [d.name for d in all_class_dirs]:
            print(f"  '{name}'")
        print("\nAdd entries to COMMON_NAME_MAP above to include more species.")
    else:
        print(f"\nMatched Lamiaceae folders ({len(matched_lc)}):")
        for folder, label, sp in sorted(matched_lc, key=lambda x: x[1]):
            n = sum(1 for f in folder.rglob("*")
                    if f.suffix.lower() in {".jpg",".jpeg",".png"})
            print(f"  '{folder.name}'  ->  {label:<35}  ({n} images)")

    print(f"\nSkipped (non-Lamiaceae): {len(skipped_lc)}")
    for name in skipped_lc[:12]:
        print(f"  {name}")
    if len(skipped_lc) > 12:
        print(f"  ... and {len(skipped_lc)-12} more")

    for folder, label, sp_name in matched_lc:
        imgs = [f for f in folder.rglob("*")
                if f.suffix.lower() in {".jpg",".jpeg",".png"}]
        for img in imgs:
            medleaf_records.append({
                "source":     "Medicinal Leaf Dataset",
                "label":      label,
                "species":    sp_name,
                "genus":      sp_name.split()[0],
                "split":      "train",
                "src_path":   str(img),
                "filename":   img.name,
                "author":     "local dataset",
                "license":    "see dataset license",
                "species_id": "",
            })

print(f"\nMedicinal Leaf Dataset  ->  {len(medleaf_records):,} images collected")
cnt_lc = Counter(r["label"] for r in medleaf_records)
for label, n in sorted(cnt_lc.items()):
    print(f"  {label:<35} {n:>5}")


## Step 7 — Merge, Deduplicate (Exact + Near-Duplicate), Pooled Labels, Splits

* **A. Identification level** — `original_name` (PlantNet species name, MedLeaves class name, or the scientific name of the local dataset's common-name folder) and `match_level` (`species` / `genus` / `unidentified`).
* **B. Exact duplicates** — SHA-256, as before.
* **C. Near-duplicates** — 64-bit pHash and dHash; a pair is a near-duplicate when pHash ≤ `PHASH_T` **and** dHash ≤ `DHASH_T`. Groups are formed by union-find; each group keeps its largest-resolution image; groups with conflicting labels are removed. Done **before** the split.
* **D. Neutral labels** — a class containing any non-exact identification becomes `<Genus>_pooled`; its members are listed in `class_composition.csv` (use this table in the paper instead of species-level interpretation).
* **E. Splits** — PlantNet keeps its official split; other sources 70/15/15 per class (seed 42).

In [ ]:
import hashlib
import imagehash

def sha256_of_file(path, chunk_size=65536):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
random.seed(RANDOM_SEED)

all_records = plantnet_records + medleaves_records + medleaf_records
assert all_records, "No images collected -- fix the paths in Step 1 and re-run Steps 4-6."
print(f"Total images collected : {len(all_records):,}")

# =============================================================================
# A. Original identification of every image and its match level
#    species = the source names exactly the target species of its class
#    genus   = another species of the same genus (routed by genus fallback)
#    unidentified = "Genus spp." / genus only
# =============================================================================
# Common names of the local Medicinal Leaf Dataset -> scientific name given by that dataset.
# Edit here if the dataset documentation identifies a folder differently.
MEDLEAF_SCIENTIFIC = {
    "Thumbe": "Leucas aspera", "Doddpathre": "Plectranthus amboinicus",
    "Mint": "Mentha sp.", "Spearmint": "Mentha spicata", "Peppermint": "Mentha x piperita",
    "Tulasi": "Ocimum tenuiflorum", "Basil": "Ocimum basilicum", "Lavender": "Lavandula sp.",
    "Rosemary": "Rosmarinus officinalis", "Thyme": "Thymus vulgaris", "Oregano": "Origanum vulgare",
    "Sage": "Salvia officinalis", "Marjoram": "Origanum majorana",
}
UNIDENTIFIED_TOKENS = {"sp", "sp.", "spp", "spp.", ""}

def original_name(r):
    if r["source"] == "PlantNet-300K":
        return r["species"]
    if r["source"] == "MedLeaves":
        try:
            return class_map[int(r["species_id"])]
        except Exception:
            return r["species"]
    folder = Path(r["src_path"]).relative_to(MEDLEAF_ROOT).parts[0]
    return MEDLEAF_SCIENTIFIC.get(folder, folder)

def match_level(name, label):
    tokens = name.replace("_", " ").split()
    if len(tokens) < 2 or tokens[1].lower() in UNIDENTIFIED_TOKENS:
        return "unidentified"
    target = ALGERIAN_LAMIACEAE.get(label, {}).get("species", "")
    t = target.split()
    same = len(t) >= 2 and tokens[0].lower() == t[0].lower() and tokens[1].lower() == t[1].lower()
    # a class that is ALREADY declared pooled (e.g. Ocimum_pooled) has no species-level members
    return "species" if same and not label.endswith("_pooled") else "genus"

for r in all_records:
    r["original_name"] = original_name(r)
    r["base_label"]    = r["label"]
    r["match_level"]   = match_level(r["original_name"], r["label"])

lvl = pd.DataFrame(all_records).groupby(["base_label", "match_level"]).size().unstack(fill_value=0)
print("\nMatch level per original class (images):")
print(lvl.to_string())

if EXCLUDE_UNIDENTIFIED:
    classes_per_genus = Counter(ALGERIAN_LAMIACEAE[l]["genus"] for l in {r["base_label"] for r in all_records}
                                if l in ALGERIAN_LAMIACEAE)
    excl = [r for r in all_records if r["match_level"] == "unidentified"
            and classes_per_genus[r["genus"]] > 1]
    excl_paths = {r["src_path"] for r in excl}
    all_records = [r for r in all_records if r["src_path"] not in excl_paths]
    print(f"\nExcluded unidentified images (genus has >1 class, cannot be assigned): {len(excl)} "
          f"{dict(Counter(r['original_name'] for r in excl))}")

# =============================================================================
# B. Exact duplicates (SHA-256)
# =============================================================================
seen_hashes, unique_records, dupes = {}, [], []
for r in tqdm(all_records, desc="SHA-256"):
    try:
        file_hash = sha256_of_file(r["src_path"])
    except (FileNotFoundError, OSError) as e:
        print(f"  could not hash {r['src_path']}: {e}")
        continue
    r["sha256"] = file_hash
    if file_hash not in seen_hashes:
        seen_hashes[file_hash] = r
        unique_records.append(r)
    else:
        original = seen_hashes[file_hash]
        dupes.append({"kept_source": original["source"], "kept_path": original["src_path"],
                      "dropped_source": r["source"], "dropped_path": r["src_path"], "sha256": file_hash})
print(f"\nExact duplicates removed (SHA-256) : {len(dupes)}")
print(f"Unique images                      : {len(unique_records):,}")
if dupes:
    pd.DataFrame(dupes).to_csv(os.path.join(OUTPUT_ROOT, "sha256_duplicates_removed.csv"), index=False)

# =============================================================================
# C. Near-duplicates (perceptual hashing): resized / cropped / recompressed copies
#    Images are grouped (union-find); each group keeps ONE image (largest resolution),
#    so no near-duplicate can end up in two splits. Groups whose members carry
#    DIFFERENT labels are removed entirely (true label unknown).
# =============================================================================
nd_log = []
if NEAR_DUP_SCREENING:
    cache_csv = os.path.join(OUTPUT_ROOT, "perceptual_hash_cache.csv")
    cache = pd.read_csv(cache_csv, dtype=str).set_index("sha256").to_dict("index") if os.path.exists(cache_csv) else {}
    new_rows = []
    for r in tqdm(unique_records, desc="Perceptual hashes"):
        c = cache.get(r["sha256"])
        if c is None:
            im = Image.open(r["src_path"]).convert("RGB")
            c = {"phash": str(imagehash.phash(im)), "dhash": str(imagehash.dhash(im)),
                 "area": str(im.size[0] * im.size[1])}
            new_rows.append({"sha256": r["sha256"], **c})
        r["phash"], r["dhash"], r["img_area"] = c["phash"], c["dhash"], int(c["area"])
    if new_rows:
        pd.concat([pd.DataFrame([{"sha256": k, **v} for k, v in cache.items()]), pd.DataFrame(new_rows)]) \
          .to_csv(cache_csv, index=False)

    PH = np.array([int(r["phash"], 16) for r in unique_records], dtype=np.uint64)
    DH = np.array([int(r["dhash"], 16) for r in unique_records], dtype=np.uint64)
    POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    def popcount(x):
        if hasattr(np, "bitwise_count"):
            return np.bitwise_count(x)
        return POP8[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)

    N, CH = len(unique_records), 256
    parent = list(range(N))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    n_pairs = 0
    for s in tqdm(range(0, N, CH), desc="Comparing hashes"):
        dp = popcount(PH[s:s + CH, None] ^ PH[None, :])
        ii, jj = np.nonzero(dp <= PHASH_T)
        sel = jj > ii + s                                   # each pair once, no self-pairs
        ii, jj = ii[sel] + s, jj[sel]
        if len(ii) == 0:
            continue
        dd = popcount(DH[ii] ^ DH[jj])
        for a, b in zip(ii[dd <= DHASH_T], jj[dd <= DHASH_T]):
            n_pairs += 1
            ra, rb = find(int(a)), find(int(b))
            if ra != rb:
                parent[ra] = rb

    groups = defaultdict(list)
    for i in range(N):
        groups[find(i)].append(i)
    keep_idx, n_dropped, n_conflict = [], 0, 0
    for gid, members in groups.items():
        for i in members:
            unique_records[i]["nd_group"] = gid if len(members) > 1 else ""
        if len(members) == 1:
            keep_idx.append(members[0]); continue
        labels = {unique_records[i]["base_label"] for i in members}
        if len(labels) > 1:
            decision, kept = "conflicting labels -> all removed", None
            n_conflict += len(members)
        else:
            kept = max(members, key=lambda i: unique_records[i]["img_area"])
            keep_idx.append(kept); decision = "kept largest"
        n_dropped += len(members) - (kept is not None)
        for i in members:
            r = unique_records[i]
            nd_log.append({"group": gid, "decision": decision, "kept": i == kept, "source": r["source"],
                           "label": r["base_label"], "original_name": r["original_name"],
                           "source_split": r["split"], "src_path": r["src_path"],
                           "phash": r["phash"], "dhash": r["dhash"], "area": r["img_area"]})
    n_groups = sum(1 for m in groups.values() if len(m) > 1)
    unique_records = [unique_records[i] for i in sorted(keep_idx)]
    nd_df = pd.DataFrame(nd_log)
    nd_df.to_csv(os.path.join(OUTPUT_ROOT, "near_duplicates_removed.csv"), index=False)
    print(f"\nNear-duplicate screening (pHash <= {PHASH_T}, dHash <= {DHASH_T}):")
    print(f"  near-duplicate pairs found          : {n_pairs:,}")
    print(f"  groups of near-duplicates           : {n_groups:,}")
    print(f"  images removed                      : {n_dropped:,} "
          f"(of which {n_conflict} in groups with conflicting labels)")
    if len(nd_df):
        cross_src = nd_df.groupby("group").source.nunique().gt(1).sum()
        cross_split = nd_df.groupby("group").source_split.nunique().gt(1).sum()
        print(f"  groups spanning >1 source           : {cross_src}")
        print(f"  groups spanning >1 original split   : {cross_split}  (potential train/test leakage, now removed)")
    print(f"  unique images after screening       : {len(unique_records):,}")
    print(f"  log -> {os.path.join(OUTPUT_ROOT, 'near_duplicates_removed.csv')}")

# =============================================================================
# D. Neutral labels for pooled classes (reviewer issue 2)
# =============================================================================
label_map = {}
for base in sorted({r["base_label"] for r in unique_records}):
    recs = [r for r in unique_records if r["base_label"] == base]
    pooled = base.endswith("_pooled") or any(r["match_level"] != "species" for r in recs)
    new = base
    if pooled and NEUTRAL_POOLED_LABELS:
        genus = ALGERIAN_LAMIACEAE[base]["genus"]
        new = f"{genus}_pooled"
        if new in label_map.values() and new != base:
            new = f"{genus}_pooled_{len(label_map)}"
    label_map[base] = new
    if new != base or base.endswith("_pooled"):
        members = Counter(r["original_name"] for r in recs)
        ALGERIAN_LAMIACEAE[new] = {**ALGERIAN_LAMIACEAE[base],
                                   "species": f"{ALGERIAN_LAMIACEAE[base]['genus']} spp. (pooled class)",
                                   "arabic": "", "fidelity": "", "region": "", "uses": "",
                                   "pooled_members": dict(members)}
for r in unique_records:
    r["label"] = label_map[r["base_label"]]
print("\nClass labels (original -> used):")
for b, n in label_map.items():
    print(f"  {b:<28} -> {n}")

comp = (pd.DataFrame(unique_records).groupby(["label", "source", "original_name", "match_level"])
        .size().rename("images").reset_index())
comp.to_csv(os.path.join(OUTPUT_ROOT, "class_composition.csv"), index=False)
print("\nClass composition (original identification of the images in each class):")
print(comp.to_string(index=False))

# =============================================================================
# E. Splits: PlantNet keeps its official split; other sources 70/15/15 per class
# =============================================================================
by_label = defaultdict(list)
for r in unique_records:
    by_label[r["label"]].append(r)
final_records = []
for label, recs in sorted(by_label.items()):
    has_split = [r for r in recs if r["source"] == "PlantNet-300K"]
    no_split  = [r for r in recs if r["source"] != "PlantNet-300K"]
    if no_split:
        random.shuffle(no_split)
        n       = len(no_split)
        n_val   = max(1, int(n * VAL_RATIO))  if n >= 6 else 0
        n_test  = max(1, int(n * VAL_RATIO))  if n >= 6 else 0
        n_train = n - n_val - n_test
        for i, r in enumerate(no_split):
            r["split"] = ("train" if i < n_train
                           else "val" if i < n_train + n_val
                           else "test")
    final_records.extend(has_split + no_split)

label_counts  = Counter(r["label"]  for r in final_records)
split_counts  = Counter(r["split"]  for r in final_records)
source_counts = Counter(r["source"] for r in final_records)
print(f"\nFINAL DATASET : {len(final_records):,} images | {len(label_counts)} classes")
print(pd.crosstab(pd.Series([r['label'] for r in final_records], name="class"),
                  pd.Series([r['split'] for r in final_records], name="split"), margins=True).to_string())

META_FIELDS = ["source", "label", "base_label", "original_name", "match_level", "species", "genus",
               "split", "filename", "author", "license", "sha256", "nd_group", "src_path", "dest_path"]


## Step 8 — Copy Files to Unified Output Structure

In [ ]:

errors    = []
skipped   = 0
print(f"Copying {len(final_records):,} images to {OUTPUT_ROOT} ...")

for r in tqdm(final_records, desc="Copying"):
    src       = Path(r["src_path"])
    safe_lbl  = re.sub(r"[^\w\-]", "_", r["label"])
    dest_dir  = Path(OUTPUT_ROOT) / r["split"] / safe_lbl
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest      = dest_dir / src.name

    if dest.exists():
        pfx  = r["source"].replace(" ","_")[:6]
        alt_dest = dest_dir / f"{pfx}_{src.name}"
        if alt_dest.exists():
            skipped += 1
            continue
        dest = alt_dest

    r["dest_path"] = str(dest)
    if not src.exists():
        errors.append(f"SRC MISSING: {src}"); continue
    try:
        shutil.copy2(src, dest)
    except Exception as e:
        errors.append(f"ERROR {src}: {e}")

meta_csv = os.path.join(OUTPUT_ROOT, "metadata.csv")
fields   = META_FIELDS   # defined in Step 7 (includes original_name, match_level, sha256, nd_group)
with open(meta_csv, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader(); w.writerows(final_records)

print(f"\nDone!  {len(final_records)-len(errors)-skipped:,} images copied")
if skipped:
    print(f"Skipped (already present as exact duplicate): {skipped}")
if errors:
    print(f"Errors ({len(errors)}):")
    for e in errors[:10]: print(f"  {e}")
print(f"metadata.csv -> {meta_csv}")


In [ ]:
import csv

meta_csv = os.path.join(OUTPUT_ROOT, "metadata.csv")

with open(meta_csv, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    final_records = list(reader)

print(f"Reloaded {len(final_records):,} records from {meta_csv}")
print("Columns:", list(final_records[0].keys()) if final_records else "none")

In [ ]:
n_collected = len(final_records)
n_unique    = len({r["src_path"] for r in final_records})
print(f"Duplicate check: {n_unique:,} unique of {n_collected:,} collected")
assert n_unique == n_collected, "Duplicates found — investigate before writing methods text!"

In [ ]:
import random
from collections import defaultdict

random.seed(42)
AUDIT_FRACTION = 0.05
review_csv = os.path.join(OUTPUT_ROOT, "manual_review_sample.csv")

# ---- Re-use the decisions already made on v2 (same source image = same decision) ----
carried = {}
prev_review = os.path.join(PREVIOUS_ROOT, "manual_review_sample.csv")
prev_meta   = os.path.join(PREVIOUS_ROOT, "metadata.csv")
if os.path.exists(prev_review) and os.path.exists(prev_meta):
    pr = pd.read_csv(prev_review)
    pm = pd.read_csv(prev_meta, encoding="utf-8-sig")[["dest_path", "src_path"]]
    pr = pr.merge(pm, on="dest_path", how="left")
    pr = pr[pr["fails_non_leaf_organ"].notna() & pr["src_path"].notna()]
    for _, row in pr.iterrows():
        carried[row["src_path"]] = (int(row["fails_non_leaf_organ"]), int(row["fails_occupancy_lt20pct"]),
                                    "" if pd.isna(row.get("notes", "")) else row.get("notes", ""))
    print(f"Previous audit decisions found: {len(carried):,} (from {prev_review})")
else:
    print("No previous audit found -- the full sample must be reviewed in the widget below.")

by_source = defaultdict(list)
for r in final_records:
    by_source[r["source"]].append(r)

sample_records = []
print(f"\n{'Source':<26}{'total':>8}{'sample':>8}{'re-used':>9}{'to review':>11}")
for source, recs in by_source.items():
    k = round(len(recs) * AUDIT_FRACTION)
    reused = [r for r in recs if r["src_path"] in carried][:k]
    rest = [r for r in recs if r["src_path"] not in carried]
    new = random.sample(rest, max(0, k - len(reused)))
    sample_records += reused + new
    print(f"{source:<26}{len(recs):>8,}{k:>8,}{len(reused):>9,}{len(new):>11,}")

with open(review_csv, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=["source", "label", "src_path", "dest_path",
                                      "fails_non_leaf_organ", "fails_occupancy_lt20pct", "notes"],
                       extrasaction="ignore")
    w.writeheader()
    for r in sample_records:
        organ, occ, notes = carried.get(r["src_path"], ("", "", ""))
        w.writerow({**r, "fails_non_leaf_organ": organ, "fails_occupancy_lt20pct": occ, "notes": notes})

n_todo = sum(1 for r in sample_records if r["src_path"] not in carried)
print(f"\nAudit sample: {len(sample_records):,} images ({len(sample_records)/len(final_records):.2%}) | "
      f"pre-filled from v2: {len(sample_records) - n_todo:,} | still to review: {n_todo:,}")
print(f"Review sheet -> {review_csv}")
print("Review every remaining image in the widget (Step 14b) so the audit covers the WHOLE sample.")


## Step 9 — Statistics Table

In [ ]:

if not final_records:
    print("[SKIP] No images in dataset — stats unavailable.")
    print("       Fix the paths in Step 1 and re-run from Step 4.")
    df = pd.DataFrame(columns=["Label","Scientific_Name","Genus",
                                "Train","Val","Test","Total",
                                "PlantNet","MedLeaves","MedLeaf_local"])
else:
    rows = []
    for label in sorted(label_counts):
        info     = ALGERIAN_LAMIACEAE.get(label, {})
        recs     = [r for r in final_records if r["label"] == label]
        by_split = Counter(r["split"]  for r in recs)
        by_src   = Counter(r["source"] for r in recs)
        rows.append({
            "Label":         label,
            "Scientific_Name": info.get("species", label),
            "Genus":         info.get("genus", ""),
            "Train":         by_split["train"],
            "Val":           by_split["val"],
            "Test":          by_split["test"],
            "Total":         label_counts[label],
            "PlantNet":      by_src.get("PlantNet-300K", 0),
            "MedLeaves":     by_src.get("MedLeaves", 0),
            "MedLeaf_local": by_src.get("Medicinal Leaf Dataset", 0),
        })
    df = pd.DataFrame(rows).sort_values("Total", ascending=False).reset_index(drop=True)
    print(f"Dataset: {len(df)} classes, {df['Total'].sum()} images total")
    print()
    print(df[["Label","Scientific_Name","Train","Val","Test","Total"]].to_string(index=False))


## Step 10 — Charts (Images per Class + Source Contribution)

In [ ]:

if df.empty:
    print("[SKIP] No data to plot.")
else:
    MIN_GOOD, MIN_OK = 50, 20
    colors = ["#1976D2" if v >= MIN_GOOD else "#FF9800" if v >= MIN_OK else "#E53935"
              for v in df["Total"]]

    n_rows = max(6, len(df) * 0.45)
    fig, axes = plt.subplots(1, 2, figsize=(18, n_rows))

    bars = axes[0].barh(df["Label"], df["Total"], color=colors)
    axes[0].set_xlabel("Number of Images", fontsize=11)
    axes[0].set_title("Images per Algerian Lamiaceae Class", fontweight="bold")
    axes[0].axvline(x=MIN_GOOD, color="gray", linestyle="--", alpha=0.6,
                    label=f">={MIN_GOOD} (target)")
    for bar, val in zip(bars, df["Total"]):
        axes[0].text(bar.get_width() + 0.5,
                     bar.get_y() + bar.get_height() / 2,
                     str(val), va="center", fontsize=8)
    handles = [mpatches.Patch(color=c, label=lbl) for c, lbl in [
        ("#1976D2", ">=50 images (good)"),
        ("#FF9800", "20–49 images (ok)"),
        ("#E53935", "<20 images (needs augmentation)"),
    ]]
    axes[0].legend(handles=handles, fontsize=9)
    axes[0].invert_yaxis()

    y_pos = list(range(len(df)))
    p1 = df["PlantNet"].tolist()
    p2 = df["MedLeaves"].tolist()
    p3 = df["MedLeaf_local"].tolist()
    axes[1].barh(y_pos, p1, color="#1976D2", label="PlantNet-300K")
    axes[1].barh(y_pos, p2, left=p1, color="#43A047", label="MedLeaves (Kaggle)")
    axes[1].barh(y_pos, p3,
                 left=[a + b for a, b in zip(p1, p2)],
                 color="#FB8C00", label="Medicinal Leaf (local)")
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(df["Label"].tolist(), fontsize=8)
    axes[1].set_xlabel("Number of Images", fontsize=11)
    axes[1].set_title("Source Contribution per Class", fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].invert_yaxis()

    plt.suptitle("Algerian Lamiaceae Leaves Dataset", fontsize=14,
                 fontweight="bold", y=1.01)
    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_ROOT, "stats_chart.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart_path}")


## Step 11 — Sample Image Grid

In [ ]:

if not final_records:
    print("[SKIP] No images to display.")
else:
    N_SAMPLES    = 5
    present      = sorted({r["label"] for r in final_records})
    n_rows       = len(present)
    n_cols       = N_SAMPLES + 1

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 2.8, n_rows * 2.8))
    if n_rows == 1:
        axes = [axes]

    for ri, label in enumerate(present):
        recs    = [r for r in final_records if r["label"] == label]
        samples = random.sample(recs, min(N_SAMPLES, len(recs)))
        info    = ALGERIAN_LAMIACEAE.get(label, {})

        axes[ri][0].axis("off")
        axes[ri][0].text(
            0.5, 0.5,
            info.get("species", label) + "\n" + info.get("arabic", ""),
            ha="center", va="center", fontsize=7,
            style="italic", wrap=True,
            transform=axes[ri][0].transAxes,
        )

        for ci, rec in enumerate(samples):
            ax = axes[ri][ci + 1]
            ax.axis("off")
            img_path = rec.get("dest_path") or rec.get("src_path", "")
            if not img_path or not Path(img_path).exists():
                img_path = rec.get("src_path", "")
            try:
                img = Image.open(img_path).convert("RGB")
                img.thumbnail((200, 200))
                ax.imshow(np.array(img))
                ax.set_title(rec["source"][:8], fontsize=5, pad=1)
            except Exception as e:
                ax.text(0.5, 0.5, "?", ha="center", va="center", fontsize=10)

        for ci in range(len(samples), N_SAMPLES):
            axes[ri][ci + 1].axis("off")

    plt.suptitle("Algerian Lamiaceae — Sample Leaf Images",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    grid_path = os.path.join(OUTPUT_ROOT, "sample_grid.png")
    plt.savefig(grid_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {grid_path}")


## Step 12 — Class Balance Report & Augmentation Recommendations

In [ ]:

if df.empty:
    print("[SKIP] No data — balance report unavailable.")
else:
    MIN_REC = 30
    print("CLASS BALANCE REPORT")
    print("=" * 65)

    low  = df[df["Total"] <  MIN_REC]
    good = df[df["Total"] >= MIN_REC]

    print(f"\nClasses with >= {MIN_REC} images : {len(good)}")
    for _, row in good.iterrows():
        print(f"  {row['Label']:<35}  {int(row['Total']):>5} images")

    print(f"\nClasses with <  {MIN_REC} images (augmentation needed) : {len(low)}")
    for _, row in low.iterrows():
        needed = MIN_REC - int(row["Total"])
        print(f"  {row['Label']:<35}  {int(row['Total']):>5} images  "
              f"(need {needed} more)")

    print("\nAugmentation recommendations:")
    print("  - Horizontal flip")
    print("  - Random rotation +-15 deg")
    print("  - Brightness / contrast jitter (+- 20 %)")
    print("  - Slight zoom (0.9 – 1.1x)")
    print("  - Gaussian noise (sigma = 0.01 – 0.03)")
    print()
    print("Thresholds for deep learning:")
    print("  - Remove classes with < 10 images")
    print("  - Minimum 30 images/class for fine-tuning pre-trained CNNs")
    print("  - Ideal: >= 100 images/class")


## Step 13 — Paper-Ready Table 1 (Methods Section)

In [ ]:

if df.empty:
    print("[SKIP] No data — Table 1 unavailable.")
else:
    t_rows = []
    for i, (_, row) in enumerate(df.sort_values("Scientific_Name").iterrows(), 1):
        info = ALGERIAN_LAMIACEAE.get(row["Label"], {})
        srcs = " + ".join(
            s for s, c in [
                ("PN300K",    row["PlantNet"]),
                ("MedLeaves", row["MedLeaves"]),
                ("MedLeaf",   row["MedLeaf_local"]),
            ] if int(c) > 0
        )
        t_rows.append({
            "No.":            i,
            "Scientific Name": info.get("species", row["Label"]),
            "Arabic Name":     info.get("arabic", ""),
            "Fidelity (FL)":   info.get("fidelity", ""),
            "Region (Algeria)": info.get("region", ""),
            "Train":           int(row["Train"]),
            "Val":             int(row["Val"]),
            "Test":            int(row["Test"]),
            "Total":           int(row["Total"]),
            "Source(s)":       srcs,
        })

    table1 = pd.DataFrame(t_rows).set_index("No.")
    print("Table 1. Algerian Lamiaceae Leaves Dataset Composition")
    print("=" * 115)
    print(table1.to_string())
    print("=" * 115)
    print(f"Total: {table1['Total'].sum()} images across {len(table1)} classes")
    print("PN300K = PlantNet-300K (Kaggle)  |  MedLeaves = Kaggle  |  MedLeaf = local")
    print("FL = Fidelity Level from Belhouala & Benarba (2021); Hassaine & Benmalek (2023)")

    path = os.path.join(OUTPUT_ROOT, "class_info.csv")
    table1.to_csv(path, encoding="utf-8-sig")
    print(f"\nSaved: {path}")
    print("Copy this table into your paper Methods section as Table 1.")


## Step 14 — Verify Output on Disk

In [ ]:

print(f"Output: {OUTPUT_ROOT}")
total = 0
for split in ["train","val","test"]:
    d = Path(OUTPUT_ROOT) / split
    if not d.exists(): continue
    cls_dirs = sorted([x for x in d.iterdir() if x.is_dir()])
    n_split  = sum(len(list(c.glob("*"))) for c in cls_dirs)
    total   += n_split
    print(f"  {split}/  ({len(cls_dirs)} classes, {n_split} images)")
    for c in cls_dirs:
        print(f"    {c.name:<35} {len(list(c.glob('*'))):>5}")

print(f"\nTotal images on disk : {total:,}")
print(f"metadata.csv         : {os.path.join(OUTPUT_ROOT,'metadata.csv')}")
print(f"class_info.csv       : {os.path.join(OUTPUT_ROOT,'class_info.csv')}")
print(f"stats_chart.png      : {os.path.join(OUTPUT_ROOT,'stats_chart.png')}")
print(f"sample_grid.png      : {os.path.join(OUTPUT_ROOT,'sample_grid.png')}")
print()
print("Dataset ready.  Set DATASET_PATH =", repr(OUTPUT_ROOT))


## Step 14b — Automated + Manual Leaf-Organ Quality Audit

Non-leaf organ images (flowers, seeds, whole plants) and images in which
the leaf occupied <20% of the frame were assessed through manual
inspection of a 5% stratified random sample from each source. A CLIP
zero-shot organ classifier (below) first flags likely non-leaf images to
speed up manual triage; the actual inclusion/exclusion decision is made by
the human reviewer in the interactive widget, not by the classifier.


In [ ]:
import os, gc
import torch
import open_clip
from PIL import Image
import pandas as pd
from tqdm import tqdm

# ---- 1. Choose the device safely (CLIP needs ~1.5 GB free GPU memory) ----
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    device = "cuda" if free_gb > 1.5 else "cpu"
    print(f"Free GPU memory: {free_gb:.2f} GB -> using {device}")
    if device == "cpu":
        print("  GPU is busy (another notebook is using it). Running on CPU -- slower but works.")
else:
    device = "cpu"
    print("No GPU -> using CPU")

# ---- 2. Load CLIP ----
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

organ_prompts = [
    "a close-up photo of a plant leaf",
    "a photo of a flower",
    "a photo of a fruit or seed pod",
    "a photo of a whole plant or shrub",
    "a photo of a plant stem or bark",
]
organ_labels = ["leaf", "flower", "fruit_seed", "whole_plant", "stem_bark"]

with torch.no_grad():
    text_features = model.encode_text(tokenizer(organ_prompts).to(device))
    text_features /= text_features.norm(dim=-1, keepdim=True)

# ---- 3. Classify all images in batches (much faster than one by one) ----
BATCH = 64 if device == "cuda" else 16
results = []
for s in tqdm(range(0, len(final_records), BATCH), desc=f"Classifying organ type ({device})"):
    batch_recs = final_records[s:s + BATCH]
    imgs, ok = [], []
    for r in batch_recs:
        try:
            imgs.append(preprocess(Image.open(r["dest_path"]).convert("RGB")))
            ok.append(True)
        except Exception:
            ok.append(False)
    probs = None
    if imgs:
        with torch.no_grad():
            f = model.encode_image(torch.stack(imgs).to(device))
            f /= f.norm(dim=-1, keepdim=True)
            probs = (100.0 * f @ text_features.T).softmax(dim=-1).cpu().numpy()
    k = 0
    for r, good in zip(batch_recs, ok):
        if good:
            p = probs[k]; k += 1
            best = int(p.argmax())
            results.append({**r, "predicted_organ": organ_labels[best],
                            "organ_confidence": float(p[best]),
                            "auto_is_leaf": organ_labels[best] == "leaf"})
        else:
            results.append({**r, "predicted_organ": "error",
                            "organ_confidence": 0.0, "auto_is_leaf": None})

# ---- 4. Save and summarise ----
df_auto = pd.DataFrame(results)
auto_csv = os.path.join(OUTPUT_ROOT, "auto_organ_classification.csv")
df_auto.to_csv(auto_csv, index=False, encoding="utf-8-sig")

n_total   = len(df_auto)
n_nonleaf = int((df_auto["auto_is_leaf"] == False).sum())
n_error   = int((df_auto["predicted_organ"] == "error").sum())
print(f"\nTotal images classified: {n_total:,}")
print(f"Flagged as non-leaf organ: {n_nonleaf:,} ({n_nonleaf / n_total:.1%})")
if n_error:
    print(f"Images that could not be read: {n_error}")
print("\nBreakdown by predicted organ:")
print(df_auto["predicted_organ"].value_counts())
print(f"\nSaved -> {auto_csv}")

# ---- 5. Free GPU memory for the next steps ----
del model; gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

review_csv = os.path.join(OUTPUT_ROOT, "manual_review_sample.csv")

df_review = pd.read_csv(review_csv)
for col in ["fails_non_leaf_organ", "fails_occupancy_lt20pct"]:
    df_review[col] = df_review[col].astype("Int64")
if "notes" not in df_review.columns:
    df_review["notes"] = ""
df_review["notes"] = df_review["notes"].fillna("")

state = {"idx": 0}

def first_unlabeled_idx():
    unlabeled = df_review.index[df_review["fails_non_leaf_organ"].isna()]
    return int(unlabeled[0]) if len(unlabeled) else 0

state["idx"] = first_unlabeled_idx()

out = widgets.Output()

img_widget   = widgets.Image(format="jpg", width=450)
progress_lbl = widgets.Label()
path_lbl     = widgets.Label()

btn_ok       = widgets.Button(description="✅ Pass (keep)", button_style="success")
btn_organ    = widgets.Button(description="❌ Non-leaf organ", button_style="danger")
btn_occ      = widgets.Button(description="❌ <20% occupancy", button_style="danger")
btn_both     = widgets.Button(description="❌ Both fail", button_style="danger")
btn_prev     = widgets.Button(description="◀ Prev")
btn_save     = widgets.Button(description="💾 Save now")

def save():
    df_review.to_csv(review_csv, index=False, encoding="utf-8-sig")

def show_current():
    with out:
        clear_output(wait=True)
        i = state["idx"]
        if i >= len(df_review):
            print("All images reviewed! Run the summary cell next.")
            return
        row = df_review.iloc[i]
        n_labeled = df_review["fails_non_leaf_organ"].notna().sum()
        progress_lbl.value = f"Image {i+1} / {len(df_review)}   |   Labeled so far: {n_labeled}/{len(df_review)}"
        path_lbl.value = f"{row['dest_path']}  [{row['source']} / {row['label']}]"
        try:
            with open(row["dest_path"], "rb") as f:
                img_widget.value = f.read()
        except Exception as e:
            print(f"Could not load image: {e}")
        display(widgets.VBox([progress_lbl, path_lbl, img_widget]))

def label_and_advance(fails_organ, fails_occ):
    i = state["idx"]
    df_review.at[i, "fails_non_leaf_organ"]    = int(fails_organ)
    df_review.at[i, "fails_occupancy_lt20pct"] = int(fails_occ)
    if i % 25 == 0:
        save()
    state["idx"] += 1
    show_current()

btn_ok.on_click(lambda b: label_and_advance(False, False))
btn_organ.on_click(lambda b: label_and_advance(True, False))
btn_occ.on_click(lambda b: label_and_advance(False, True))
btn_both.on_click(lambda b: label_and_advance(True, True))
btn_prev.on_click(lambda b: (state.update(idx=max(0, state["idx"]-1)), show_current()))
btn_save.on_click(lambda b: (save(), print("Saved.")))

controls = widgets.HBox([btn_ok, btn_organ, btn_occ, btn_both, btn_prev, btn_save])
display(controls, out)
show_current()

In [ ]:

import numpy as np

df_manual_full = pd.read_csv(review_csv)
n_total_sample = len(df_manual_full)

df_manual = df_manual_full[df_manual_full["fails_non_leaf_organ"].notna()].copy()
n = len(df_manual)

if n == 0:
    print("=" * 70)
    print("REVIEW NOT STARTED")
    print("=" * 70)
    print(f"0 of {n_total_sample:,} sampled images have been labeled yet.")
    print("Complete the review in the widget above, then re-run this cell.")
elif n < n_total_sample:
    pct = n / n_total_sample
    print("=" * 70)
    print("REVIEW IN PROGRESS")
    print("=" * 70)
    print(f"{n:,} of {n_total_sample:,} images labeled so far ({pct:.1%}).")
    print("Statistics below are PROVISIONAL — based on the partial sample.")
    print("Re-run this cell after the review widget shows 'All images")
    print("reviewed!' to get the final, citable numbers.\n")
else:
    print(f"All {n_total_sample:,} sampled images have been reviewed.\n")

if n > 0:
    df_manual["failed"] = (
        (df_manual["fails_non_leaf_organ"] == 1) |
        (df_manual["fails_occupancy_lt20pct"] == 1)
    )
    x = int(df_manual["failed"].sum())
    p = x / n

    z = 1.96
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    ci_low, ci_high = max(0.0, center - margin), min(1.0, center + margin)

    print("=" * 70)
    print("FINAL MANUAL QUALITY AUDIT")
    print("=" * 70)
    print(f"Images reviewed : {n:,}")
    print(f"Images failed   : {x:,} ({p:.1%}; 95% CI: {ci_low:.1%}-{ci_high:.1%}, Wilson score)")

    print()
    print("Failure rate by source:")
    source_stats = df_manual.groupby("source")["failed"].agg(reviewed="size", failed="sum")
    source_stats["rate"] = source_stats["failed"] / source_stats["reviewed"]
    for source, row in source_stats.iterrows():
        print(f"  {source:<30} {int(row['failed']):>3}/{int(row['reviewed']):<4} ({row['rate']:.2%})")

    print()
    print("Failure rate by class:")
    class_stats = df_manual.groupby("label")["failed"].agg(reviewed="size", failed="sum")
    for label, row in class_stats.iterrows():
        if row["failed"] > 0:
            print(f"  {label:<30} {int(row['failed']):>3}/{int(row['reviewed']):<4}")

    failing_sources = sorted(df_manual.loc[df_manual["failed"], "source"].unique())
    print()
    if len(failing_sources) == 1:
        print(f"All {x} failing images originated from {failing_sources[0]}.")
    elif failing_sources:
        print(f"Failing images originated from: {', '.join(failing_sources)}")
    else:
        print("No failing images found in the reviewed sample.")

    print()
    print("This rate reflects the pre-correction audit finding and provides an")
    print("estimate of the prevalence of residual filtering errors in the broader")
    print("dataset, including the uninspected images, since full manual inspection")
    print("of all images was not performed. The flagged images are removed from")
    print("the released dataset in the next cell.")

In [ ]:

import os, shutil

df_manual = pd.read_csv(review_csv)
bad_images = df_manual[
    (df_manual["fails_non_leaf_organ"] == 1) | (df_manual["fails_occupancy_lt20pct"] == 1)
]
print(f"{len(bad_images)} images identified for removal:")
print(bad_images[["source", "label", "dest_path"]].to_string(index=False))

removed_dir = os.path.join(OUTPUT_ROOT, "_removed_flagged_images")
os.makedirs(removed_dir, exist_ok=True)

moved = 0
for path in bad_images["dest_path"]:
    if os.path.exists(path):
        dest = os.path.join(removed_dir, os.path.basename(path))
        shutil.move(path, dest)
        moved += 1
print(f"\nMoved {moved} images to {removed_dir}")

n_before = len(final_records)
bad_dest_paths = set(bad_images["dest_path"])
final_records[:] = [r for r in final_records if r["dest_path"] not in bad_dest_paths]
label_counts  = Counter(r["label"]  for r in final_records)
split_counts  = Counter(r["split"]  for r in final_records)
source_counts = Counter(r["source"] for r in final_records)

print(f"Final corrected dataset : {len(final_records):,} images "
      f"(removed {n_before - len(final_records)} of {n_before:,})")

meta_csv_clean = os.path.join(OUTPUT_ROOT, "metadata_cleaned.csv")
fields = META_FIELDS
with open(meta_csv_clean, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader(); w.writerows(final_records)
print(f"Saved cleaned metadata -> {meta_csv_clean}")


## Step 15 — Class Imbalance Fix: Augmentation to 1,000 images/class

Starting from the audit-corrected dataset (images flagged in the manual
quality audit above removed), several classes still have far fewer training images
than others. This cell augments every class that has fewer than 1,000
training images to reach exactly 1,000 by generating synthetic images
using randomised transforms. Classes that already have >= 1,000 training
images are left untouched. Only train/ is modified — val/ and test/ are
never touched by this cell. Exact per-class before/after counts are printed
by the cell below; re-run Step 9 or Step 18 afterwards to refresh the
summary tables and charts.

**Transforms applied:** horizontal flip, rotation +/-20 deg, brightness/contrast
jitter, zoom 0.85-1.15x, horizontal/vertical shift +/-10%, Gaussian noise.
Each augmented image gets a unique filename so it never overwrites originals.


In [ ]:

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "Pillow", "--quiet"])

import os, random, math
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np

TARGET_N   = 1000
TRAIN_DIR  = Path(OUTPUT_ROOT) / "train"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


def aug_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def aug_rotate(img):
    angle = random.uniform(-20, 20)
    return img.rotate(angle, expand=False,
                      fillcolor=_edge_color(img))

def aug_brightness(img):
    factor = random.uniform(0.7, 1.3)
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img):
    factor = random.uniform(0.75, 1.25)
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_zoom(img):
    """Random crop then resize back to original dimensions."""
    w, h  = img.size
    scale = random.uniform(0.85, 1.00)
    nw, nh = int(w * scale), int(h * scale)
    x0 = random.randint(0, w - nw)
    y0 = random.randint(0, h - nh)
    return img.crop((x0, y0, x0 + nw, y0 + nh)).resize((w, h), Image.LANCZOS)

def aug_shift(img):
    """Translate image by up to 10% in each axis."""
    w, h   = img.size
    dx     = random.randint(-int(w * 0.10), int(w * 0.10))
    dy     = random.randint(-int(h * 0.10), int(h * 0.10))
    shifted = Image.new(img.mode, (w, h), _edge_color(img))
    shifted.paste(img, (dx, dy))
    return shifted

def aug_noise(img):
    arr  = np.array(img, dtype=np.float32)
    noise = np.random.normal(0, random.uniform(3, 10), arr.shape)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def aug_saturation(img):
    factor = random.uniform(0.6, 1.4)
    return ImageEnhance.Color(img).enhance(factor)

def _edge_color(img):
    """Return dominant border pixel color for fill (avoids black bars)."""
    arr = np.array(img)
    border = np.concatenate([
        arr[0, :].reshape(-1, arr.shape[2]),
        arr[-1, :].reshape(-1, arr.shape[2]),
        arr[:, 0].reshape(-1, arr.shape[2]),
        arr[:, -1].reshape(-1, arr.shape[2]),
    ])
    mean = border.mean(axis=0).astype(int)
    return tuple(mean[:3])

TRANSFORMS = [aug_flip, aug_rotate, aug_brightness,
              aug_contrast, aug_zoom, aug_shift,
              aug_noise, aug_saturation]

def augment_one(img):
    """Apply 2-4 random transforms to one image."""
    k   = random.randint(2, 4)
    ops = random.sample(TRANSFORMS, k)
    out = img.copy()
    for op in ops:
        try:
            out = op(out)
        except Exception:
            pass
    return out


if not TRAIN_DIR.exists():
    print(f"[ERROR] Training directory not found: {TRAIN_DIR}")
else:
    class_dirs = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
    summary    = []

    for cls_dir in class_dirs:
        imgs = sorted([f for f in cls_dir.iterdir()
                       if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        n_orig = len(imgs)

        if n_orig >= TARGET_N:
            summary.append((cls_dir.name, n_orig, 0, n_orig))
            continue

        n_needed  = TARGET_N - n_orig
        generated = 0

        source_cycle = imgs * (math.ceil(n_needed / max(n_orig, 1)) + 1)
        random.shuffle(source_cycle)

        print(f"  {cls_dir.name:<35} {n_orig:>5} -> augmenting +{n_needed}...",
              end="", flush=True)

        for i, src_path in enumerate(source_cycle[:n_needed]):
            try:
                with Image.open(src_path).convert("RGB") as base:
                    aug = augment_one(base)
                    out_name = f"aug_{i:05d}_{src_path.stem}.jpg"
                    aug.save(cls_dir / out_name, "JPEG", quality=92)
                    generated += 1
            except Exception as e:
                print(f"\n    [WARN] {src_path.name}: {e}")

        summary.append((cls_dir.name, n_orig, generated, n_orig + generated))
        print(f" done ({n_orig + generated} total)")

    print()
    print("=" * 60)
    print(f"  {'Class':<33} {'Before':>7} {'Added':>7} {'After':>7}")
    print("=" * 60)
    for name, before, added, after in summary:
        flag = "" if after >= TARGET_N else " <"
        print(f"  {name:<33} {before:>7} {added:>7} {after:>7}{flag}")
    print("=" * 60)
    total_after = sum(a for *_, a in summary)
    print(f"  {'TOTAL':<33} {'':>7} {'':>7} {total_after:>7}")
    print()
    print("Augmentation complete. Re-run Step 9 (stats) to update the table.")


## Step 16 — Undersample the *Lamium* (pooled) class in train/

The *Lamium* class (now `Lamium_pooled`, 11 PlantNet *Lamium* taxa) remains by far the largest class in train/.
This cell randomly keeps `LAMIUM_CAP` images and moves the rest to `train_lamium_backup/` so they are never
deleted and can be restored. The exact before/after counts and the resulting imbalance ratio are printed below.

In [ ]:

import random, shutil
from pathlib import Path

LAMIUM_CAP  = 5000
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

TRAIN_DIR  = Path(OUTPUT_ROOT) / "train"
lamium_dirs = sorted(d for d in TRAIN_DIR.iterdir() if d.is_dir() and d.name.startswith("Lamium"))
LAMIUM_DIR = lamium_dirs[0] if lamium_dirs else TRAIN_DIR / "Lamium_pooled"
BACKUP_DIR = Path(OUTPUT_ROOT) / "train_lamium_backup"

if not LAMIUM_DIR.exists():
    print(f"[SKIP] {LAMIUM_DIR} not found")
else:
    all_imgs = sorted([f for f in LAMIUM_DIR.iterdir()
                       if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    n_total  = len(all_imgs)
    print(f"{LAMIUM_DIR.name} train images : {n_total}")

    if n_total <= LAMIUM_CAP:
        print(f"Already <= {LAMIUM_CAP}. Nothing to do.")
    else:
        random.shuffle(all_imgs)
        keep    = set(f.name for f in all_imgs[:LAMIUM_CAP])
        to_move = [f for f in all_imgs if f.name not in keep]

        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        for f in to_move:
            shutil.move(str(f), BACKUP_DIR / f.name)

        remaining = len([f for f in LAMIUM_DIR.iterdir()
                         if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        print(f"Kept in train/{LAMIUM_DIR.name} : {remaining}")
        print(f"Moved to backup                : {len(to_move)}")
        print(f"Backup folder                  : {BACKUP_DIR}")
        print()

    print("Updated train/ counts:")
    counts = {}
    for d in sorted(TRAIN_DIR.iterdir()):
        if not d.is_dir():
            continue
        n = len([f for f in d.iterdir()
                 if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        counts[d.name] = n
        bar = "#" * (n // 200)
        print(f"  {d.name:<35} {n:>6}  {bar}")

    total = sum(counts.values())
    print(f"  {'TOTAL':<35} {total:>6}")

    nonzero = [n for n in counts.values() if n > 0]
    if len(nonzero) >= 2:
        ratio = max(nonzero) / min(nonzero)
        print(f"\nImbalance ratio: {ratio:.1f}x  (train split)")
    else:
        print("\n[INFO] Not enough non-empty classes to compute an imbalance ratio.")


## Step 17 -- (DISABLED) Augment val/ and test/ for *Leucas aspera*

**Disabled by default.** This step used to inflate *Leucas aspera*'s val/test
sets to `LEUCAS_VAL_TARGET` images by synthetic augmentation. That
contradicts the Methods text, which states val/test are "preserved at their
natural size" for every class. The original markdown here also cited "22
images in each of val/test" -- that figure was itself already double the
true raw count (11 each), a symptom of the OUTPUT_ROOT duplication bug fixed
in Cell 2/17.

If you want augmented val/test for Leucas after all, re-enable the cell
below and update Table 1 + the Methods text to report 100/100 (or whatever
`LEUCAS_VAL_TARGET` is set to) instead of the raw 11/11.


In [ ]:

ENABLE_LEUCAS_VAL_TEST_AUGMENTATION = False

if not ENABLE_LEUCAS_VAL_TEST_AUGMENTATION:
    print("[SKIPPED] Leucas val/test augmentation is disabled -- val/test "
          "stay at their natural (raw) size for every class, matching the "
          "current Methods text.")
else:
    import random, math
    from pathlib import Path
    from PIL import Image
    import numpy as np

    LEUCAS_VAL_TARGET = 100
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)


    OUTPUT = Path(OUTPUT_ROOT)

    for split in ["val", "test"]:
        leucas_dir = OUTPUT / split / "Leucas_aspera"
        if not leucas_dir.exists():
            print(f"[SKIP] {leucas_dir} not found")
            continue

        imgs = sorted([f for f in leucas_dir.iterdir()
                       if f.suffix.lower() in {".jpg",".jpeg",".png"}])
        n_orig    = len(imgs)
        n_needed  = max(0, LEUCAS_VAL_TARGET - n_orig)

        if n_needed == 0:
            print(f"  {split}/Leucas_aspera already >= {LEUCAS_VAL_TARGET}. Skipping.")
            continue

        print(f"  {split}/Leucas_aspera : {n_orig} -> augmenting +{n_needed}...",
              end="", flush=True)

        cycle  = imgs * (math.ceil(n_needed / max(n_orig,1)) + 1)
        random.shuffle(cycle)
        generated = 0

        for i, src_path in enumerate(cycle[:n_needed]):
            try:
                with Image.open(src_path).convert("RGB") as base:
                    aug = augment_one(base)
                    out_name = f"aug_{split}_{i:04d}_{src_path.stem}.jpg"
                    aug.save(leucas_dir / out_name, "JPEG", quality=92)
                    generated += 1
            except Exception as e:
                print(f"\n    [WARN] {src_path.name}: {e}")

        print(f" done ({n_orig + generated} total)")

    print()
    print("Updated val/ and test/ Leucas_aspera counts:")
    for split in ["val", "test"]:
        d = OUTPUT / split / "Leucas_aspera"
        if d.exists():
            n = len([f for f in d.iterdir()
                     if f.suffix.lower() in {".jpg",".jpeg",".png"}])
            print(f"  {split}/Leucas_aspera : {n}")

    print()
    print("Dataset finalised. Run Step 9 (statistics) to regenerate the summary table.")


## Step 18 — Final Dataset Statistics & Visualisations

Counts files directly from disk — reflects all augmentation and undersampling steps.

In [ ]:

import os, random
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

OUTPUT_DIR = Path(OUTPUT_ROOT)
IMG_EXT    = {".jpg", ".jpeg", ".png"}

meta_clean_path = OUTPUT_DIR / "metadata_cleaned.csv"
if meta_clean_path.exists():
    with open(meta_clean_path, encoding="utf-8-sig") as _f:
        baseline_n = sum(1 for _ in _f) - 1
    print(f"Audit-corrected baseline (post-removal, pre-augmentation): {baseline_n:,} images")
    print("(images flagged in the 5% manual quality audit were removed; see Step 14b)")
    print()

rows = []
for split in ["train", "val", "test"]:
    split_dir = OUTPUT_DIR / split
    if not split_dir.exists():
        continue
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXT)
        rows.append({"split": split, "label": cls_dir.name, "n": n})

df_raw = pd.DataFrame(rows)

if df_raw.empty:
    print("[ERROR] No images found. Check OUTPUT_ROOT path.")
else:
    df = df_raw.pivot(index="label", columns="split", values="n").fillna(0).astype(int)
    for col in ["train", "val", "test"]:
        if col not in df.columns:
            df[col] = 0
    df["Total"] = df["train"] + df["val"] + df["test"]
    df = df.sort_values("train", ascending=False)[["train", "val", "test", "Total"]]

    SEP = "=" * 70
    print(SEP)
    print("  ALGERIAN LAMIACEAE DATASET — FINAL STATISTICS")
    print(SEP)
    print(f"  {'Class':<33} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 70)
    for label, row in df.iterrows():
        sp = label.replace("_", " ")
        print(f"  {sp:<33} {row['train']:>7,} {row['val']:>7,} {row['test']:>7,} {row['Total']:>8,}")
    print("-" * 70)
    tot = df.sum()
    print(f"  {'TOTAL':<33} {tot['train']:>7,} {tot['val']:>7,} {tot['test']:>7,} {tot['Total']:>8,}")
    print(SEP)
    ratio = df["train"].max() / df["train"].min()
    print(f"\n  Classes         : {len(df)}")
    print(f"  Imbalance ratio : {ratio:.1f}x  (train split)")
    print(f"  Train / Val / Test : {tot['train']/tot['Total']*100:.1f}% / "
          f"{tot['val']/tot['Total']*100:.1f}% / "
          f"{tot['test']/tot['Total']*100:.1f}%")

    class_labels = [s.replace("_", " ") for s in df.index]
    x = np.arange(len(class_labels))
    w = 0.25
    split_colors = {"train": "#1976D2", "val": "#43A047", "test": "#FB8C00"}

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for i, split in enumerate(["train", "val", "test"]):
        axes[0].bar(x + i*w, df[split], width=w,
                    label=split.capitalize(),
                    color=split_colors[split], alpha=0.88)
    axes[0].set_xticks(x + w)
    axes[0].set_xticklabels(class_labels, rotation=35, ha="right", fontsize=9)
    axes[0].set_ylabel("Number of images")
    axes[0].set_title("Images per class per split", fontweight="bold")
    axes[0].legend()
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))

    thresholds = [(5000, "#1976D2"), (1000, "#43A047"), (0, "#FB8C00")]
    bar_colors = []
    for val in df["train"]:
        for thr, col in thresholds:
            if val >= thr:
                bar_colors.append(col)
                break

    bars = axes[1].barh(class_labels, df["train"], color=bar_colors)
    axes[1].set_xlabel("Train images")
    axes[1].set_title("Training set — image counts", fontweight="bold")
    axes[1].invert_yaxis()
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
    for bar, val in zip(bars, df["train"]):
        axes[1].text(bar.get_width() + 30,
                     bar.get_y() + bar.get_height() / 2,
                     f"{val:,}", va="center", fontsize=9)

    patch_g = mpatches.Patch(color="#1976D2", label=">= 5000 images")
    patch_m = mpatches.Patch(color="#43A047", label="1000 – 4999 images")
    patch_l = mpatches.Patch(color="#FB8C00", label="< 1000 images")
    axes[1].legend(handles=[patch_g, patch_m, patch_l], fontsize=9)

    plt.suptitle("Algerian Lamiaceae Leaves — Final Dataset",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    chart_path = OUTPUT_DIR / "final_stats_chart.png"
    plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nChart saved: {chart_path}")

    print("Generating sample grid...")
    N = 5
    labels_list = list(df.index)
    fig2, axes2 = plt.subplots(len(labels_list), N + 1,
                                figsize=((N+1)*2.8, len(labels_list)*2.8))
    if len(labels_list) == 1:
        axes2 = [axes2]

    for ri, label in enumerate(labels_list):
        train_cls = OUTPUT_DIR / "train" / label
        all_imgs = ([f for f in train_cls.iterdir()
                     if f.suffix.lower() in IMG_EXT]
                    if train_cls.exists() else [])
        originals = [f for f in all_imgs if not f.name.startswith("aug_")]
        pool = originals if len(originals) >= N else all_imgs
        samples = random.sample(pool, min(N, len(pool)))

        axes2[ri][0].axis("off")
        axes2[ri][0].text(0.5, 0.5,
                          label.replace("_", " "),
                          ha="center", va="center",
                          fontsize=7, style="italic",
                          transform=axes2[ri][0].transAxes)

        for ci, img_path in enumerate(samples):
            ax = axes2[ri][ci + 1]
            ax.axis("off")
            try:
                img = Image.open(img_path).convert("RGB")
                img.thumbnail((200, 200))
                ax.imshow(np.array(img))
            except Exception:
                ax.text(0.5, 0.5, "?", ha="center", va="center")

        for ci in range(len(samples), N):
            axes2[ri][ci + 1].axis("off")

    
    csv_path = OUTPUT_DIR / "final_class_info.csv"
    df.index.name = "Class"
    df.to_csv(str(csv_path), encoding="utf-8-sig")
    print(f"CSV saved: {csv_path}")


In [ ]:

meta_clean_path = OUTPUT_DIR / "metadata_cleaned.csv"
assert meta_clean_path.exists(), (
    "metadata_cleaned.csv not found — run Step 14b's removal cell first "
    "(the final stats below would otherwise reflect the UNCORRECTED, "
    "pre-audit dataset)."
)

with open(meta_clean_path, encoding="utf-8-sig") as _f:
    baseline_n = sum(1 for _ in _f) - 1
print(f"Audit-corrected baseline confirmed: {baseline_n:,} images "
      f"(from {meta_clean_path.name})")

n_train_on_disk = sum(1 for f in (OUTPUT_DIR / "train").rglob("*")
                       if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert n_train_on_disk > baseline_n * 0.5, (
    f"train/ on disk has only {n_train_on_disk:,} images, which looks too "
    f"low relative to the {baseline_n:,}-image baseline — did augmentation "
    f"(Step 15) and undersampling (Step 16) actually run? Re-run Steps "
    f"15-17 before trusting this table."
)
print(f"train/ on disk: {n_train_on_disk:,} images — looks post-augmentation, proceeding.\n")

In [ ]:
import pandas as pd, os

OUTPUT_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"   # works even after a kernel restart

meta = pd.read_csv(os.path.join(OUTPUT_ROOT, "metadata.csv"), encoding="utf-8-sig")   # 23,179 after near-dup
nd   = pd.read_csv(os.path.join(OUTPUT_ROOT, "near_duplicates_removed.csv"))          # near-duplicate groups
label_map = dict(zip(meta["base_label"], meta["label"]))

after_nd = meta.groupby("label").size().rename("After near-dup")
removed  = (nd[~nd["kept"]].assign(label=lambda d: d["label"].map(label_map))
            .groupby("label").size().rename("Near-dup removed"))
table = pd.concat([after_nd, removed], axis=1).fillna(0).astype(int)
table.insert(0, "After SHA-256", table["After near-dup"] + table["Near-dup removed"])
table.loc["TOTAL"] = table.sum()
print(table.to_string())
print(f"\nCheck: after SHA-256 total = {table.loc['TOTAL', 'After SHA-256']:,} (should be 24,455)")

In [1]:
import os, pandas as pd, numpy as np
OUTPUT_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
review_csv  = os.path.join(OUTPUT_ROOT, "manual_review_sample.csv")